# 06 Leakage audit and split design (Phase 6)
No model is trained. Protocol: `configs/evaluation_protocol.yaml`, `reports/evaluation_protocol.md`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
R = ROOT / "results/audit"
M = ROOT / "data/manifests"
pd.set_option("display.max_rows", 80); pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("project root:", ROOT.name)

project root: research_paper


In [2]:
LK = pd.read_csv(R / 'leakage_findings.csv')
LK.groupby(['category','severity']).size()

category                        severity
context_feature_review          info         5
                                medium       3
duplicate_leakage_across_split  high         3
                                medium      23
label_defining_feature          medium      15
never_a_model_input             critical     3
                                high         6
single_feature_separability     info         4
                                medium      25
dtype: int64

In [3]:
SP = pd.read_csv(R / 'feature_separability.csv')
SP[SP.variant == 'improved'].sort_values('ovl_benign_vs_attack')[['dataset_family','feature','role','ovl_benign_vs_attack','ovl_attack_vs_attempted']].head(25)

,dataset_family,feature,role,ovl_benign_vs_attack,ovl_attack_vs_attempted
98,CSE-CIC-IDS-2018,Bwd Packet Length Max,feature,0.092451,0.041228
43,CIC-IDS-2017,Packet Length Max,feature,0.095052,0.462959
7,CIC-IDS-2017,Total Length of Bwd Packet,feature,0.095817,0.468383
12,CIC-IDS-2017,Bwd Packet Length Max,feature,0.099719,0.468222
129,CSE-CIC-IDS-2018,Packet Length Max,feature,0.099998,0.038350
87,CSE-CIC-IDS-2018,Dst Port,context_review,0.102622,0.050551
58,CIC-IDS-2017,Bwd Segment Size Avg,feature,0.108363,0.478556
14,CIC-IDS-2017,Bwd Packet Length Mean,feature,0.108363,0.478556
44,CIC-IDS-2017,Packet Length Mean,feature,0.112734,0.488313
56,CIC-IDS-2017,Average Packet Size,feature,0.112734,0.488313


In [4]:
pd.read_csv(R / 'split_design_coverage.csv')

,protocol,dataset_family,partition,n_rows,n_attack,n_attempted,attack_families,min_ts,max_ts,n_files,primary_evidence
0,P0_random_rows,CIC-IDS-2017,test,315111,76200,1819,"[""Botnet"", ""DDoS"", ""DoS GoldenEye"", ""DoS Hulk""...",2017-07-03 11:56:41.708215,2017-07-07 20:02:38.574765,5,diagnostic only: never primary evidence for te...
1,P0_random_rows,CIC-IDS-2017,train,1470468,353924,8423,"[""Botnet"", ""DDoS"", ""DoS GoldenEye"", ""DoS Hulk""...",2017-07-03 11:55:58.598308,2017-07-07 20:02:40.977798,5,diagnostic only: never primary evidence for te...
2,P0_random_rows,CIC-IDS-2017,val,314397,75307,1737,"[""Botnet"", ""DDoS"", ""DoS GoldenEye"", ""DoS Hulk""...",2017-07-03 11:56:37.908639,2017-07-07 20:02:23.415694,5,diagnostic only: never primary evidence for te...
3,P0_random_rows,CSE-CIC-IDS-2018,excluded,2609,0,0,[],2018-02-21 12:33:55.396864,2018-02-22 12:21:47.198334,1,diagnostic only: never primary evidence for te...
4,P0_random_rows,CSE-CIC-IDS-2018,test,9477184,530038,45952,"[""Botnet Ares"", ""DDoS-HOIC"", ""DDoS-LOIC-HTTP"",...",2018-02-14 12:28:09.698070,2018-03-03 00:39:24.738032,10,diagnostic only: never primary evidence for te...
5,P0_random_rows,CSE-CIC-IDS-2018,train,44234877,2473953,214228,"[""Botnet Ares"", ""DDoS-HOIC"", ""DDoS-LOIC-HTTP"",...",2018-02-14 12:28:07.743746,2018-03-03 00:39:53.894472,10,diagnostic only: never primary evidence for te...
6,P0_random_rows,CSE-CIC-IDS-2018,val,9480475,531431,46057,"[""Botnet Ares"", ""DDoS-HOIC"", ""DDoS-LOIC-HTTP"",...",2018-02-14 12:28:08.383839,2018-03-03 00:39:16.730889,10,diagnostic only: never primary evidence for te...
7,P1_within_day_chronological,CIC-IDS-2017,embargo,37203,7667,111,"[""Heartbleed"", ""Infiltration"", ""Infiltration -...",2017-07-03 16:45:10.111026,2017-07-07 18:28:16.841741,5,"evaluated, NOT recommended as primary: 2018 te..."
8,P1_within_day_chronological,CIC-IDS-2017,test,416138,121576,750,"[""DDoS"", ""Heartbleed"", ""Infiltration"", ""Infilt...",2017-07-03 18:26:38.144423,2017-07-07 20:02:40.977798,5,"evaluated, NOT recommended as primary: 2018 te..."
9,P1_within_day_chronological,CIC-IDS-2017,train,1098469,176446,9898,"[""Botnet"", ""DoS GoldenEye"", ""DoS Hulk"", ""DoS S...",2017-07-03 11:55:58.598308,2017-07-07 16:47:20.056273,5,"evaluated, NOT recommended as primary: 2018 te..."


In [5]:
pd.read_csv(R / 'split_duplicate_leakage.csv')

,protocol,dataset_family,eval_partition,n_rows,n_rows_feature_vector_seen_in_train,share_seen_in_train,n_attack,n_attack_seen_in_train,share_attack_seen_in_train,n_rows_seen_in_train_or_val,share_seen_in_train_or_val,n_attack_seen_in_train_or_val,share_attack_seen_in_train_or_val
0,P0_random_rows,CIC-IDS-2017,val,314397,48885,0.155488,75307,19627,0.260627,NaN,NaN,NaN,NaN
1,P0_random_rows,CIC-IDS-2017,test,315111,49096,0.155805,76200,19671,0.258150,52468.0,0.166506,21046.0,0.276194
2,P0_random_rows,CSE-CIC-IDS-2018,val,9480475,2840387,0.299604,531431,6356,0.011960,NaN,NaN,NaN,NaN
3,P0_random_rows,CSE-CIC-IDS-2018,test,9477184,2841096,0.299783,530038,6349,0.011978,2871544.0,0.302995,6595.0,0.012443
4,P1_within_day_chronological,CIC-IDS-2017,val,548166,38242,0.069764,199742,0,0.000000,NaN,NaN,NaN,NaN
5,P1_within_day_chronological,CIC-IDS-2017,test,416138,33508,0.080521,121576,0,0.000000,50508.0,0.121373,12365.0,0.101706
6,P1_within_day_chronological,CSE-CIC-IDS-2018,val,10982688,3247703,0.295711,5123,2407,0.469842,NaN,NaN,NaN,NaN
7,P1_within_day_chronological,CSE-CIC-IDS-2018,test,834247,237316,0.284467,0,0,NaN,240438.0,0.288210,0.0,NaN
8,P1b_within_day_flow_quantile,CIC-IDS-2017,val,364698,52915,0.145093,76881,20897,0.271810,NaN,NaN,NaN,NaN
9,P1b_within_day_flow_quantile,CIC-IDS-2017,test,391879,37303,0.095190,76800,1623,0.021133,45619.0,0.116411,4044.0,0.052656


In [6]:
FL = pd.read_csv(R / 'split_family_leakage.csv')
FL[FL.eval_partition == 'test'][['protocol','family','n_attack','family_in_train','family_in_val','share_seen_in_train','share_seen_in_train_or_val']]

,protocol,family,n_attack,family_in_train,family_in_val,share_seen_in_train,share_seen_in_train_or_val
15,P0_random_rows,Botnet,116,True,True,0.000000,0.000000
16,P0_random_rows,DDoS,14317,True,True,0.000000,0.000000
17,P0_random_rows,DoS GoldenEye,1124,True,True,0.000000,0.000000
18,P0_random_rows,DoS Hulk,24014,True,True,0.000000,0.000000
19,P0_random_rows,DoS Slowhttptest,249,True,True,0.000000,0.000000
20,P0_random_rows,DoS Slowloris,608,True,True,0.000000,0.000000
21,P0_random_rows,FTP-Patator,596,True,True,0.000000,0.000000
22,P0_random_rows,Heartbleed,4,True,True,0.000000,0.000000
23,P0_random_rows,Infiltration,6,True,True,0.000000,0.000000
24,P0_random_rows,Infiltration - Portscan,10888,True,True,0.549963,0.573567


In [7]:
pd.read_csv(R / 'label_conflict_composition.csv').groupby('dataset_family').head(5)

,dataset_family,label_set,n_groups,n_rows,max_files_per_group,rows_per_label,n_min_error_rows,feature_key
0,CIC-IDS-2017,Infiltration - Portscan | Portscan,4368,14118,2,"{""Infiltration - Portscan"": 6762, ""Portscan"": ...",4939,feature + context_review columns (as duplicate...
1,CIC-IDS-2017,BENIGN | Portscan,7,22,2,"{""BENIGN"": 8, ""Portscan"": 14}",7,feature + context_review columns (as duplicate...
2,CSE-CIC-IDS-2018,BENIGN | FTP-BruteForce - Attempted,9,263736,2,"{""BENIGN"": 60, ""FTP-BruteForce - Attempted"": 2...",60,feature + context_review columns (as duplicate...
3,CSE-CIC-IDS-2018,BENIGN | Infiltration - NMAP Portscan,4,36,4,"{""BENIGN"": 7, ""Infiltration - NMAP Portscan"": 29}",5,feature + context_review columns (as duplicate...
